# Step 3: Data Enrichment

Goal: Add derived fields and map to operational schema.

New fields:
- experienceLevel (from yearsCode)
- usesDocumentation (from learningMethods)
- usesAIForLearning (from learningMethods)
- usesStackOverflow (from learningMethods)

In [4]:
import pandas as pd
import json
import os

In [5]:
# Create my_work folder if it doesn't exist
os.makedirs("/my_work", exist_ok=True)

## Load Cleaned Data

In [6]:
# Load from JSONL
records = []
with open("../my_work/cleaned_data.jsonl", "r") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

In [7]:
df.shape

(49191, 11)

In [8]:
df.head()

,ResponseId,Age,YearsCode,DevType,LearnCodeChoose,LearnCode,LearnCodeAI,AILearnHow,AISelect,AIAcc,AISent
0,1,25-34 years old,14.0,"Developer, mobile","Yes, I am not new to coding but am learning ne...",[Online Courses or Certification (includes all...,"Yes, I learned how to use AI-enabled tools for...",[AI CodeGen tools or AI-enabled apps],"Yes, I use AI tools monthly or infrequently",Neither trust nor distrust,Indifferent
1,2,25-34 years old,10.0,"Developer, back-end","Yes, I am not new to coding but am learning ne...",[Online Courses or Certification (includes all...,"Yes, I learned how to use AI-enabled tools for...",[AI CodeGen tools or AI-enabled apps],"Yes, I use AI tools weekly",Neither trust nor distrust,Indifferent
2,3,35-44 years old,12.0,"Developer, front-end","Yes, I am not new to coding but am learning ne...",[Online Courses or Certification (includes all...,"Yes, I learned how to use AI-enabled tools for...","[AI CodeGen tools or AI-enabled apps, Technica...","Yes, I use AI tools daily",Somewhat trust,Favorable
3,4,35-44 years old,5.0,"Developer, back-end","Yes, I am not new to coding but am learning ne...","[Other online resources (e.g. standard search,...","Yes, I learned how to use AI-enabled tools for...","[AI CodeGen tools or AI-enabled apps, Videos (...","Yes, I use AI tools weekly",Somewhat trust,Favorable
4,5,35-44 years old,22.0,Engineering manager,"No, I am not new to coding and did not learn n...",None,"Yes, I learned how to use AI-enabled tools for...",[Technical documentation (is generated for/by ...,"Yes, I use AI tools weekly",Neither trust nor distrust,Favorable


## Define Constants

These are the exact option texts from the survey.

**Where did these come from?**
From Step 1 (exploration) - we saw these exact strings in the LearnCode column.

**Why use constants?**
To avoid typos.

In [9]:
TECH_DOC = "Technical documentation (is generated for/by the tool or system)"
AI_CODEGEN = "AI CodeGen tools or AI-enabled apps"
STACK_OVERFLOW = "Stack Overflow or Stack Exchange"

## Function 1: Calculate Experience Level

**What it does:**
Takes years of coding and returns a category.

**Business rules:**
- 0-2 years = Beginner
- 3-5 years = Early Career
- 6-10 years = Experienced
- 11+ years = Highly Experienced
- Missing (None or pd.NA) = Unknown

**Critical:** Check for missing values FIRST using pd.isna()!

In [10]:
def experience_level(years_code):
    # Check for missing values (None or pd.NA)
    if pd.isna(years_code):
        return "Unknown"
    if years_code <= 2:
        return "Beginner"
    if years_code <= 5:
        return "Early Career"
    if years_code <= 10:
        return "Experienced"
    return "Highly Experienced"

In [11]:
# Test
experience_level(1)

'Beginner'

In [12]:
experience_level(None)

'Unknown'

In [13]:
experience_level(pd.NA)

'Unknown'

## Function 2: Check if Option is in List

**What it does:**
Checks if a specific learning method was selected.

**Example:**
```python
learningMethods = ["Books", "Stack Overflow"]
has_option(learningMethods, "Stack Overflow")  # True
has_option(None, "Books")                      # False
```

In [14]:
def has_option(options, target):
    if options is None:
        return False
    return target in options

In [15]:
# Test
has_option(["Books", "Stack Overflow"], "Stack Overflow")

True

In [16]:
has_option(None, "Books")

False

## Add Derived Columns

Now we add the 4 new columns using apply().

In [17]:
# Add experienceLevel
df['experienceLevel'] = df['YearsCode'].apply(experience_level)

In [18]:
# Add boolean flags
df['usesDocumentation'] = df['LearnCode'].apply(lambda x: has_option(x, TECH_DOC))
df['usesAIForLearning'] = df['LearnCode'].apply(lambda x: has_option(x, AI_CODEGEN))
df['usesStackOverflow'] = df['LearnCode'].apply(lambda x: has_option(x, STACK_OVERFLOW))

In [19]:
# Check what we have
df.columns

Index(['ResponseId', 'Age', 'YearsCode', 'DevType', 'LearnCodeChoose',
       'LearnCode', 'LearnCodeAI', 'AILearnHow', 'AISelect', 'AIAcc', 'AISent',
       'experienceLevel', 'usesDocumentation', 'usesAIForLearning',
       'usesStackOverflow'],
      dtype='str')

In [20]:
df[['YearsCode', 'experienceLevel']].head(10)

,YearsCode,experienceLevel
0,14.0,Highly Experienced
1,10.0,Experienced
2,12.0,Highly Experienced
3,5.0,Early Career
4,22.0,Highly Experienced
5,20.0,Highly Experienced
6,13.0,Highly Experienced
7,30.0,Highly Experienced
8,15.0,Highly Experienced
9,10.0,Experienced


In [21]:
df['experienceLevel'].value_counts()

experienceLevel
Highly Experienced    25817
Experienced           10365
Unknown                6149
Early Career           5153
Beginner               1707
Name: count, dtype: int64

## Rename Columns (Map to Operational Schema)

**Field mapping:**
- ResponseId -> responseId
- LearnCode -> learningMethods
- AILearnHow -> aiLearningMethods
- AISelect -> aiUsage
- AIAcc -> aiTrust
- AISent -> aiSentiment

In [22]:
df = df.rename(columns={
    'ResponseId': 'responseId',
    'Age': 'age',
    'YearsCode': 'yearsCode',
    'DevType': 'devType',
    'LearnCodeChoose': 'learnCodeChoose',
    'LearnCode': 'learningMethods',
    'LearnCodeAI': 'learnCodeAI',
    'AILearnHow': 'aiLearningMethods',
    'AISelect': 'aiUsage',
    'AIAcc': 'aiTrust',
    'AISent': 'aiSentiment',
})

In [23]:
# Check columns
df.columns

Index(['responseId', 'age', 'yearsCode', 'devType', 'learnCodeChoose',
       'learningMethods', 'learnCodeAI', 'aiLearningMethods', 'aiUsage',
       'aiTrust', 'aiSentiment', 'experienceLevel', 'usesDocumentation',
       'usesAIForLearning', 'usesStackOverflow'],
      dtype='str')

In [24]:
# Should be 15 columns
len(df.columns)

15

In [25]:
df.head()

,responseId,age,yearsCode,devType,learnCodeChoose,learningMethods,learnCodeAI,aiLearningMethods,aiUsage,aiTrust,aiSentiment,experienceLevel,usesDocumentation,usesAIForLearning,usesStackOverflow
0,1,25-34 years old,14.0,"Developer, mobile","Yes, I am not new to coding but am learning ne...",[Online Courses or Certification (includes all...,"Yes, I learned how to use AI-enabled tools for...",[AI CodeGen tools or AI-enabled apps],"Yes, I use AI tools monthly or infrequently",Neither trust nor distrust,Indifferent,Highly Experienced,False,False,False
1,2,25-34 years old,10.0,"Developer, back-end","Yes, I am not new to coding but am learning ne...",[Online Courses or Certification (includes all...,"Yes, I learned how to use AI-enabled tools for...",[AI CodeGen tools or AI-enabled apps],"Yes, I use AI tools weekly",Neither trust nor distrust,Indifferent,Experienced,False,False,True
2,3,35-44 years old,12.0,"Developer, front-end","Yes, I am not new to coding but am learning ne...",[Online Courses or Certification (includes all...,"Yes, I learned how to use AI-enabled tools for...","[AI CodeGen tools or AI-enabled apps, Technica...","Yes, I use AI tools daily",Somewhat trust,Favorable,Highly Experienced,True,False,False
3,4,35-44 years old,5.0,"Developer, back-end","Yes, I am not new to coding but am learning ne...","[Other online resources (e.g. standard search,...","Yes, I learned how to use AI-enabled tools for...","[AI CodeGen tools or AI-enabled apps, Videos (...","Yes, I use AI tools weekly",Somewhat trust,Favorable,Early Career,True,True,True
4,5,35-44 years old,22.0,Engineering manager,"No, I am not new to coding and did not learn n...",None,"Yes, I learned how to use AI-enabled tools for...",[Technical documentation (is generated for/by ...,"Yes, I use AI tools weekly",Neither trust nor distrust,Favorable,Highly Experienced,False,False,False


## Save to JSONL

In [26]:
with open("../my_work/processed_data.jsonl", "w") as f:
    for _, row in df.iterrows():
        record = row.to_dict()
        
        # Convert NaN to None (skip lists)
        for key, value in record.items():
            if not isinstance(value, list) and pd.isna(value):
                record[key] = None
        
        f.write(json.dumps(record) + "\n")

## Next Step

Run validation:
```bash
cd Day4_Project
python3 validation/check_step2_processed.py my_work/processed_data.jsonl
```

---

## Important Notes

### Why use pd.isna() instead of checking None?

When working with Pandas, missing values can be:
- `None` (Python)
- `pd.NA` (Pandas nullable type)
- `float('nan')` (NumPy)

`pd.isna()` catches all of them:
```python
pd.isna(None)         # True
pd.isna(pd.NA)        # True
pd.isna(float('nan')) # True
pd.isna(5)            # False
```

But checking `is None` only catches Python None:
```python
# WRONG - misses pd.NA
if years_code is None:
    return "Unknown"

# CORRECT - catches all missing values
if pd.isna(years_code):
    return "Unknown"
```

### Using apply() with Pandas

```python
# Apply function to each value
df['newCol'] = df['oldCol'].apply(my_function)

# Apply with lambda
df['newCol'] = df['oldCol'].apply(lambda x: my_function(x, arg))
```